# Wafer Defect RCA — Demo

Walks through the three-agent pipeline on one MixedWM38 wafer map:
**Detection (CLIP zero-shot) -> Hybrid Retrieval -> Diagnosis (GPT-4o-mini)**.

Prereqs: run `data/download_mixedwm38.py`, `data/build_corpus.py`, and
`knowledge_base/build_index.py` first. The diagnosis step needs a funded
`OPENAI_API_KEY` in `.env`; detection + retrieval run without one.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from config import settings
from agents.detection_agent import wafer_array_to_pil

In [ ]:
# Load a sample wafer map and display it
data = np.load(os.path.join(settings.mixedwm38_data_dir, 'data.npy'))
labels = np.load(os.path.join(settings.mixedwm38_data_dir, 'labels.npy'))
idx = int(np.where(labels.argmax(1) == 1)[0][0])  # a Center example
wafer = data[idx]
plt.imshow(wafer_array_to_pil(wafer)); plt.axis('off'); plt.title('wafer map'); plt.show()

In [ ]:
# 1) Detection
from agents.detection_agent import DetectionAgent
detection_agent = DetectionAgent()
detection = detection_agent.detect(wafer)
print('defect_type:', detection['defect_type'], '| confidence: %.3f' % detection['confidence'])
print('active_defects:', detection['active_defects'])

In [ ]:
# 2) Retrieval (optionally enriched with an engineer note)
from agents.retrieval_agent import RetrievalAgent
retrieval_agent = RetrievalAgent()
retrieval = retrieval_agent.retrieve(detection, engineer_note='ring pattern after CMP step')
print('query:', retrieval['query_used'])
for d in retrieval['retrieved_docs']:
    print(' -', d['source'], '::', d['text'][:80])

In [ ]:
# 3) Full pipeline via LangGraph (needs a funded OPENAI_API_KEY for diagnosis)
from agents.diagnosis_agent import DiagnosisAgent
from pipeline.graph import build_graph
graph = build_graph(detection_agent, retrieval_agent, DiagnosisAgent())
np.save('_demo.npy', wafer)
state = {k: None for k in ['detection_result','detection_latency_ms','retrieval_result',
    'retrieval_latency_ms','diagnosis_result','diagnosis_latency_ms','total_latency_ms','error_message']}
state.update({'wafer_image_path': '_demo.npy', 'engineer_note': 'ring pattern after CMP step',
              'pipeline_status': 'running'})
final = graph.invoke(state)
print('status:', final['pipeline_status'])
print(json.dumps(final.get('diagnosis_result'), indent=2, default=str))